# 1 · Corpus and cleaning

The parallel corpus is 30 Indian High Court judgments, each present as OCR'd
English and its Hindi translation. This notebook checks what the OCR actually
handed us, shows what the cleaning pipeline changes, and applies a quality gate, because **five of the thirty Hindi files are not Hindi at all**.

Nothing is defined in the notebook: the rules live in `court_mt.corpus.*` and are
applied once by `scripts/01_build_corpus.py`, so every later stage reads the same
text.

In [1]:
from court_mt.notebook import setup

theme = setup()

## What the raw export looks like

The `clean/*.txt` files are OCR output, not prose. Two artefacts matter for
tokenisation: lines are hard-wrapped at roughly 55 characters, and blank lines
appear *inside* list items rather than only between paragraphs.

In [2]:
import pandas as pd

from court_mt import paths
from court_mt.corpus.loader import find_clean_dirs

clean_dirs = find_clean_dirs(paths.RAW)
raw_hindi = (clean_dirs["hi"] / "1.txt").read_text(encoding="utf-8")

print(f"{len(raw_hindi.splitlines())} lines, {len(raw_hindi)} characters\n")
print("\n".join(repr(line) for line in raw_hindi.splitlines()[:8]))

162 lines, 9232 characters

'\ufeff1. अपीलार्थी ने मुजफ्फरनगर क्षेत्रिय ग्रामीण बैंक (इसके बाद \'बैंक" में क्षेत्रीय ग्रामीण'
'बैंक सेवाओं के एक अधिकारी के रूप में काम किया)। 27.05.2003 को, उन्हें'
'आरोप-पत्र दिया गया और पंद्रह दिनों की अवधि के भीतर अपना जवाब प्रस्तुत'
'करने के लिए कहा गया। अपीलकर्ता के खिलाफ लगाए गए आरोप निम्न हैं: '
''
'1. आपने प्रधान कार्यालय के निर्देशों और विनियमों का पालन नहीं किया।'
''
'2. आपने बैंक के हित के विरूद्ध/हानिकारक काम किया है और आप'


## What cleaning does

`clean()` reflows the hard wrapping into logical paragraphs, NFC-normalises
(which matters for Devanagari, decomposed matras tokenise differently from
composed ones), strips invisible characters including the leading BOM visible
above, and folds typographic punctuation to ASCII.

The reflow rule is the interesting one. A blank line is *not* a paragraph break
here, because the OCR scatters blank lines inside list items; a new paragraph
starts at an explicit list marker, or after a blank line only when the preceding
text actually ended a sentence.

In [3]:
from court_mt.corpus.clean import clean

cleaned_hindi = clean(raw_hindi)
paragraphs = cleaned_hindi.split("\n\n")

print(f"{len(raw_hindi.splitlines())} wrapped lines -> {len(paragraphs)} paragraphs\n")
for paragraph in paragraphs[:4]:
    print("¶", paragraph[:150], "...\n")

162 wrapped lines -> 23 paragraphs

¶ 1. अपीलार्थी ने मुजफ्फरनगर क्षेत्रिय ग्रामीण बैंक (इसके बाद 'बैंक" में क्षेत्रीय ग्रामीण बैंक सेवाओं के एक अधिकारी के रूप में काम किया)। 27.05.2003 को ...

¶ 1. आपने प्रधान कार्यालय के निर्देशों और विनियमों का पालन नहीं किया। ...

¶ 2. आपने बैंक के हित के विरूद्ध/हानिकारक काम किया है और आप अनधिकृत रूप से अनुपस्थित थे। ...

¶ 3. आप बैंक के हित और आपने अपने कर्तव्यों से भिन्न कार्य किया हैं। ...



Character count barely moves, which is the point: cleaning changes *structure*,
not content. Anything removing a meaningful share of characters would be silently
editing the corpus.

In [4]:
for name, side in (("hindi", raw_hindi), ("english", (clean_dirs["en"] / "1.txt").read_text(encoding="utf-8"))):
    out = clean(side)
    delta = len(out) - len(side)
    print(f"{name:8s} {len(side):6,d} -> {len(out):6,d} chars  ({delta:+,d}, {delta / len(side):+.2%})")

hindi     9,232 ->  9,214 chars  (-18, -0.19%)
english   9,999 ->  9,977 chars  (-22, -0.22%)


## The quality gate: five files are not Hindi

Length-based alignment checks pass every pair in this corpus. They are not
enough. Five Hindi files had their Devanagari destroyed at **encoding** time, not by bad OCR, and survive only as runs of `?`:

    1.???????????. 1181(??/??)???? 1998?????????????????????????

Their file sizes are normal, their hi/en length ratios are normal, and their
English sides are perfectly intact. They are invisible to any check that looks at
length rather than script.

They cannot be left in. A run of `?` is cheap and uniform to tokenise in *every*
vocabulary, so these documents pull all the tokenizers toward each other and
compress exactly the differences the study exists to measure.

In [5]:
from court_mt.corpus.loader import load_judgments
from court_mt.corpus.validate import assess

all_pairs = [
    type(j)(doc_id=j.doc_id, en=clean(j.en), hi=clean(j.hi))
    for j in load_judgments(paths.RAW)
]
quality = pd.DataFrame([vars(assess(j)) for j in all_pairs])

quality[~quality.usable].reset_index(drop=True)

INFO    court_mt.corpus.loader: loaded 30 paired judgments from /Users/karan/Desktop/adalat-court-mt/data/raw


,doc_id,hi_chars,devanagari_share,replacement_share,usable,reason
0,6,4790,0.000,0.924,False,only 0.0% Devanagari (expected >30%)
1,14,5731,0.000,0.938,False,only 0.0% Devanagari (expected >30%)
2,22,5043,0.000,0.915,False,only 0.0% Devanagari (expected >30%)
3,25,12112,0.000,0.915,False,only 0.0% Devanagari (expected >30%)
4,26,4301,0.000,0.929,False,only 0.0% Devanagari (expected >30%)


In [6]:
corrupted = quality[~quality.usable].doc_id.tolist()
sample = next(j for j in all_pairs if j.doc_id == corrupted[0])

print(f"doc {sample.doc_id} — english side is intact:")
print(" ", sample.en[:150], "...\n")
print(f"doc {sample.doc_id} — hindi side is not:")
print(" ", sample.hi[:150], "...")

doc 6 — english side is intact:
  1. Feeling aggrieved and dissatisfied with the impugned judgment and order dated 16.07.2014 passed by the High Court of Judicature at Allahabad (Luckn ...

doc 6 — hindi side is not:
  1.???????????. 1181(??/??)???? 1998?????????????????????????(????????)??????????? 16.07.2014?????????????????????????????????????????????????????????? ...


### Could the source PDFs recover them?

Partially, but not cleanly. The Hindi PDFs do contain Devanagari, yet their font
CMaps are broken, so extraction returns text peppered with control characters
(`प्रति\x03वेद्य`). That is measurably worse than the 25 intact files, and mixing
two text qualities into one corpus would confound every comparison downstream.

Quarantining is the honest call, and `scripts/01_build_corpus.py` writes the
rejects to `data/processed/quarantine.jsonl` so the decision stays auditable.

In [7]:
from pypdf import PdfReader

from court_mt.tokenization.metrics import count_devanagari

pdf_path = clean_dirs["hi"].parent / "original" / f"{corrupted[0]}.pdf"
extracted = "\n".join((page.extract_text() or "") for page in PdfReader(str(pdf_path)).pages)

print(f"{len(extracted):,} chars extracted, {count_devanagari(extracted) / len(extracted):.1%} Devanagari")
print("control characters present:", sorted({c for c in extracted if ord(c) < 32 and c not in '\n\t'}))
print()
print(repr(extracted[:120]))

8,583 chars extracted, 53.8% Devanagari
control characters present: ['\x03', '\x0e', '\x16']

'प्रति\x03वेद्य\nभार\x03ीय सव\x0eच्च न्यायालय\nसिसविवल अपीलीय अति\x16कारिर\x03ा\nसिसविवल अपील संख्या 6945 वर्ष  2021\nउ.प्र. राज्य\n... अपीला'


## The processed corpus

`scripts/01_build_corpus.py` applies cleaning and the gate, then writes
`data/processed/corpus.jsonl`. Everything downstream reads that file.

In [8]:
from court_mt.corpus.dataset import load_corpus

judgments = load_corpus()
print(f"{len(judgments)} usable pairs of {len(all_pairs)} (quarantined: {', '.join(corrupted)})")

25 usable pairs of 30 (quarantined: 6, 14, 22, 25, 26)


In [9]:
from court_mt.tokenization.metrics import words

stats = pd.DataFrame([
    {
        "doc_id": j.doc_id,
        "en_words": len(words(j.en)),
        "hi_words": len(words(j.hi)),
        "en_chars": len(j.en),
        "hi_chars": len(j.hi),
        "hi_devanagari_share": count_devanagari(j.hi) / len(j.hi),
        "hi_en_char_ratio": len(j.hi) / len(j.en),
    }
    for j in judgments
])

print(stats[["en_words", "hi_words", "en_chars", "hi_chars"]].sum().to_string(), "\n")
stats.describe().loc[["mean", "min", "50%", "max"]]

en_words     37503
hi_words     39058
en_chars    216773
hi_chars    202738 



,en_words,hi_words,en_chars,hi_chars,hi_devanagari_share,hi_en_char_ratio
mean,1500.120,1562.320,8670.920,8109.520,0.754,0.937
min,773.000,786.000,4274.000,4217.000,0.727,0.872
50%,1497.000,1548.000,8616.000,7727.000,0.756,0.935
max,2921.000,3129.000,17283.000,16213.000,0.772,1.014


With the corrupted files gone, the hi/en length ratio tightens from 0.73, 1.01 to
0.87, 1.01, independent confirmation that the quarantined pairs were the
outliers, and that what remains is genuinely parallel.

In [10]:
ratio = stats.hi_en_char_ratio
print(f"hi/en character ratio: median {ratio.median():.2f}, range {ratio.min():.2f}-{ratio.max():.2f}")
print(f"Devanagari share of Hindi text: {stats.hi_devanagari_share.min():.1%}-{stats.hi_devanagari_share.max():.1%}")

hi/en character ratio: median 0.93, range 0.87-1.01
Devanagari share of Hindi text: 72.7%-77.2%


Hindi runs slightly *shorter* than English in characters (median ratio 0.93),
which is worth holding on to: Devanagari packs more meaning per character. So any
tokenizer charging more for Hindi than for English is paying a pure encoding
penalty, not translating a longer text.

That penalty is what notebook 2 measures.